# Jena Climate — RNN **v2**: Otimização de Hiperparâmetros (expandida)

> **O que é este notebook.** É a versão *v2* focada **exclusivamente na otimização de hiperparâmetros** (*hyperparameter tuning*). No notebook principal (`jena_climate_rnn_all_features`) a pesquisa de hiperparâmetros resumia-se a um único varrimento do **número de unidades** (32/64/128). Aqui esse processo é **expandido** para cobrir os hiperparâmetros mais relevantes de uma RNN de regressão, com uma metodologia documentada e reprodutível.

**Curso:** Aprendizagem Profunda (Deep Learning) — Mestrado em Engenharia Informática, ISEP 2025/2026

---

### O que muda em relação ao notebook principal
- **Pipeline de dados e modelo:** é **exatamente o mesmo** (variante com **todas as 14 variáveis** meteorológicas → 19 *features* após codificação cíclica). Reproduz-se aqui de forma condensada apenas para o notebook ser **autónomo** (corre do início ao fim sem depender do outro).
- **Foco:** em vez de treinar a matriz de 8 arquiteturas, este notebook fixa a **arquitetura líder (GRU @ 24 h)** e faz uma pesquisa sistemática sobre os seus hiperparâmetros.

### Hiperparâmetros pesquisados
| Hiperparâmetro | Papel | Valores explorados |
|---|---|---|
| `units` | Capacidade da camada recorrente | 32 – 256 |
| `learning_rate` | Passo do otimizador Adam | 3e-4 – 1e-2 |
| `dropout` | Regularização (anti-*overfitting*) | 0.0 – 0.5 |
| `batch_size` | Nº de amostras por atualização | 64 – 512 |
| `dense_units` | Tamanho da camada densa-ponte | 16 – 64 |

### Estratégia de pesquisa (3 níveis)
1. **OFAT** (*One-Factor-At-A-Time*) — varia-se **um** hiperparâmetro de cada vez, mantendo os restantes no valor *baseline*. Mostra a **sensibilidade** isolada de cada fator. (Secção 3.2–3.5)
2. **Grid search** — varrimento exaustivo do produto cartesiano dos **2 fatores mais influentes** (`units` × `learning_rate`), visualizado como *heatmap*. Captura **interações** que o OFAT não vê. (Secção 3.6)
3. **Random search** — amostragem aleatória do espaço completo (5 hiperparâmetros em simultâneo). É a forma mais eficiente de explorar espaços de dimensão alta. (Secção 3.7)

Por fim, a **melhor configuração** encontrada é **revalidada com 3 sementes** e comparada com o *baseline* para verificar se o ganho é **real** ou apenas **ruído de inicialização** (Secção 4).


---
## 0. Setup & Imports

In [ ]:
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print(f'TensorFlow   : {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

### Métrica RMSLE (mesma convenção do notebook principal)

O RMSLE usa `log1p(x) = log(1 + x)`, indefinido para `x ≤ −1`. Como a temperatura varia entre ~ −23 °C e +37 °C, o RMSLE é calculado no **domínio escalado [0, 1]** (após o MinMaxScaler), aplicando `clip([0, 1])` apenas no momento da avaliação. As **métricas de decisão** continuam a ser o **RMSE e o MAE em °C** (fisicamente interpretáveis); o RMSLE é reportado por exigência do enunciado.

In [ ]:
def rmsle(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(tf.math.log1p(y_pred) - tf.math.log1p(y_true))))

print('RMSLE metric defined.')

---
## 1. Carregamento & Pré-processamento (condensado)

Bloco idêntico ao do notebook principal (variante com **todas as variáveis**). Está aqui apenas para tornar o notebook autónomo — a justificação detalhada de cada passo (codificação cíclica, *downsampling* horário, *split* cronológico, normalização ajustada só ao treino) está documentada no notebook principal. O resultado é o dicionário `datasets`, que alimenta toda a pesquisa de hiperparâmetros.

In [ ]:
DATA_PATH = 'jena_climate_2009_2016.csv'
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date Time'] = pd.to_datetime(df_raw['Date Time'], format='%d.%m.%Y %H:%M:%S')
df_raw = df_raw.set_index('Date Time').sort_index()
print(f'Shape: {df_raw.shape}  |  {df_raw.index[0]}  ->  {df_raw.index[-1]}')

In [ ]:
# Feature selection (VARIANTE: manter TODAS as variaveis) + bad-value handling
RAW_FEATURES = ['T (degC)', 'p (mbar)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)',
                'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)',
                'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']
TARGET = 'T (degC)'

df = df_raw[RAW_FEATURES].copy()
for col in ['wv (m/s)', 'max. wv (m/s)']:        # -9999 = erro de sensor
    df[col] = df[col].replace(-9999.0, np.nan)
df = df.ffill().bfill()
print(f'Missing apos limpeza: {df.isnull().sum().sum()}  |  features (raw): {df.shape[1]}')

In [ ]:
# Codificacao ciclica (direcao do vento + hora + dia-do-ano) e downsampling horario
df['wd_sin'] = np.sin(2 * np.pi * df['wd (deg)'] / 360.0)
df['wd_cos'] = np.cos(2 * np.pi * df['wd (deg)'] / 360.0)
df = df.drop(columns=['wd (deg)'])
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24.0)
df['doy_sin']  = np.sin(2 * np.pi * df.index.dayofyear / 365.0)
df['doy_cos']  = np.cos(2 * np.pi * df.index.dayofyear / 365.0)

df_hourly = df.resample('h').mean().dropna()
print(f'Registos horarios: {len(df_hourly):,}  |  features: {df_hourly.shape[1]}')

In [ ]:
# Split cronologico 70/15/15 + normalizacao MinMax (ajustada SO ao treino)
N = len(df_hourly)
TRAIN_END, VAL_END = int(N * 0.70), int(N * 0.85)
train_df = df_hourly.iloc[:TRAIN_END]
val_df   = df_hourly.iloc[TRAIN_END:VAL_END]
test_df  = df_hourly.iloc[VAL_END:]

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df).astype(np.float32)
val_scaled   = scaler.transform(val_df).astype(np.float32)
test_scaled  = scaler.transform(test_df).astype(np.float32)

TARGET_IDX = list(df_hourly.columns).index(TARGET)
N_FEATURES = df_hourly.shape[1]
print(f'Train/Val/Test = {len(train_df):,}/{len(val_df):,}/{len(test_df):,}  '
      f'|  features={N_FEATURES}  target_idx={TARGET_IDX}')

In [ ]:
# Construcao de sequencias (janela deslizante) e datasets
HORIZON = 24   # prever a temperatura 24 h no futuro

def create_sequences(data, window_size, horizon=HORIZON):
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i: i + window_size])
        y.append(data[i + window_size + horizon - 1, TARGET_IDX])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# O tuning foca-se na janela de 24 h (configuracao lider). A janela de 48 h fica
# disponivel caso se queira incluir o tamanho da janela na pesquisa.
WINDOW_SIZES = {'24h': 24, '48h': 48}
datasets = {}
for name, W in WINDOW_SIZES.items():
    X_tr, y_tr = create_sequences(train_scaled, W)
    X_va, y_va = create_sequences(val_scaled,   W)
    X_te, y_te = create_sequences(test_scaled,  W)
    datasets[name] = {'window': W, 'X_train': X_tr, 'y_train': y_tr,
                      'X_val': X_va, 'y_val': y_va, 'X_test': X_te, 'y_test': y_te}
    print(f'{name:>4}: train X{X_tr.shape}  val X{X_va.shape}  test X{X_te.shape}')

---
## 2. Modelo & Funções Auxiliares (preparadas para tuning)

A função `build_model` é a **mesma** do notebook principal, mas agora expõe como argumentos **todos** os hiperparâmetros que vamos pesquisar (`units`, `dropout`, `lr` e ainda `dense_units` — o tamanho da camada densa-ponte, que antes estava fixo em 32).

A novidade é a função **`run_single(...)`**: treina + avalia **uma** configuração com **uma** semente e devolve uma **linha** (dicionário) com os hiperparâmetros e as métricas de teste. Todas as pesquisas (OFAT, grid, random) reduzem-se a chamar repetidamente esta função e a empilhar as linhas numa tabela `DataFrame`.

In [ ]:
SEEDS = [42, 7, 123]

def set_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_model(arch, window_size, units=64, dropout=0.2, lr=1e-3, dense_units=32):
    # Mesma arquitetura do notebook principal; dense_units passa a ser tunable.
    model = Sequential(name=f'{arch}_w{window_size}h_u{units}')
    input_shape = (window_size, N_FEATURES)
    rnn_layers = {'SimpleRNN': layers.SimpleRNN, 'LSTM': layers.LSTM, 'GRU': layers.GRU}
    if arch in rnn_layers:
        model.add(rnn_layers[arch](units, input_shape=input_shape, return_sequences=False))
    elif arch == 'BiLSTM':
        model.add(layers.Bidirectional(
            layers.LSTM(units, return_sequences=False), input_shape=input_shape))
    else:
        raise ValueError(f'Unknown architecture: {arch}')
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.Dense(1, activation='linear'))   # saida linear (regressao)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='mse', metrics=['mae'])
    return model


def train_model(model, ds, epochs, batch_size):
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0),
    ]
    return model.fit(ds['X_train'], ds['y_train'],
                     validation_data=(ds['X_val'], ds['y_val']),
                     epochs=epochs, batch_size=batch_size, callbacks=callbacks, verbose=0)


def evaluate_model(model, ds):
    X_te, y_te = ds['X_test'], ds['y_test']
    y_pred_sc = model.predict(X_te, verbose=0).flatten()
    # inverte a normalizacao para voltar a graus Celsius
    dummy_true = np.zeros((len(y_te), N_FEATURES), dtype=np.float32)
    dummy_pred = np.zeros((len(y_te), N_FEATURES), dtype=np.float32)
    dummy_true[:, TARGET_IDX] = y_te
    dummy_pred[:, TARGET_IDX] = y_pred_sc
    y_true_c = scaler.inverse_transform(dummy_true)[:, TARGET_IDX]
    y_pred_c = scaler.inverse_transform(dummy_pred)[:, TARGET_IDX]
    rmse_c = float(np.sqrt(mean_squared_error(y_true_c, y_pred_c)))
    mae_c  = float(mean_absolute_error(y_true_c, y_pred_c))
    rmsle_val = float(rmsle(tf.constant(np.clip(y_te, 0.0, 1.0), dtype=tf.float32),
                            tf.constant(np.clip(y_pred_sc, 0.0, 1.0), dtype=tf.float32)).numpy())
    return {'rmse_c': rmse_c, 'mae_c': mae_c, 'rmsle_scaled': rmsle_val}

print('Helpers prontos: build_model / train_model / evaluate_model')

In [ ]:
EPOCHS_TUNE = 80   # limite superior; o early stopping costuma parar antes

def run_single(arch, window, units=64, dropout=0.2, lr=1e-3, batch_size=256,
               dense_units=32, seed=42, epochs=EPOCHS_TUNE, verbose=True):
    # Treina + avalia UMA configuracao com UMA semente.
    # Devolve uma linha (dict) hiperparametros + metricas, pronta p/ DataFrame.
    set_seeds(seed)
    ds = datasets[window]
    t0 = time.time()
    mdl  = build_model(arch, ds['window'], units=units, dropout=dropout,
                       lr=lr, dense_units=dense_units)
    hist = train_model(mdl, ds, epochs=epochs, batch_size=batch_size)
    met  = evaluate_model(mdl, ds)
    row = {'arch': arch, 'window': window, 'units': units, 'dropout': dropout,
           'lr': lr, 'batch_size': batch_size, 'dense_units': dense_units, 'seed': seed,
           'params': int(mdl.count_params()), 'epochs_run': len(hist.history['loss']),
           'RMSE (C)': round(met['rmse_c'], 3), 'MAE (C)': round(met['mae_c'], 3),
           'RMSLE (scaled)': round(met['rmsle_scaled'], 4), 'time_s': round(time.time() - t0, 1)}
    if verbose:
        print(f"  {arch} u={units:>3} drop={dropout} lr={lr:<6g} bs={batch_size:>3} "
              f"dense={dense_units:>2} -> RMSE={row['RMSE (C)']:.3f}C "
              f"MAE={row['MAE (C)']:.3f}C ({row['epochs_run']} ep, {row['time_s']}s)")
    return row

print('run_single() pronto.')

---
## 3. Pesquisa de Hiperparâmetros (expandida)

### Metodologia

- **Arquitetura/janela fixas:** todo o tuning é feito sobre o **GRU @ 24 h**, a configuração líder identificada no notebook principal. Fixar a arquitetura torna a pesquisa comparável e mantém o custo computacional controlado.
- **Semente fixa (42) durante a pesquisa:** ao procurar a melhor combinação, fixa-se a semente para que as diferenças observadas se devam aos **hiperparâmetros** e não ao acaso da inicialização. Só no fim (Secção 4) a melhor configuração é revalidada com **várias sementes**.
- **Banda de ruído (`NOISE_BAND ≈ 0.13 °C`):** valor herdado do notebook principal (desvio-padrão típico do RMSE entre sementes a 24 h). **Qualquer diferença de RMSE inferior a esta banda não é significativa** — é apenas ruído de inicialização. Esta é a régua honesta para interpretar todos os resultados abaixo.

> ⚠️ **Custo computacional.** Este notebook treina ~50 modelos no total (1 baseline + 17 OFAT + 9 grid + 15 random + 12 na validação final). Em CPU pode demorar bastante; com GPU é confortável. Reduz `N_TRIALS` / o tamanho das grelhas, ou `EPOCHS_TUNE`, se precisares de um ensaio mais rápido. **Executa secção a secção.**

In [ ]:
# ---- Constantes e utilitarios da pesquisa ----------------------------------
TUNE_ARCH   = 'GRU'     # arquitetura lider (notebook principal)
TUNE_WINDOW = '24h'     # janela lider
NOISE_BAND  = 0.13      # +/- C : ruido de inicializacao entre sementes (do nb principal)

# Hiperparametros baseline = configuracao usada em todo o notebook principal
BASELINE = dict(units=64, dropout=0.2, lr=1e-3, batch_size=256, dense_units=32)

def ofat(param, values):
    # Varia UM hiperparametro mantendo os restantes no baseline (One-Factor-At-A-Time).
    rows = []
    for v in values:
        kw = dict(BASELINE)
        kw[param] = v
        rows.append(run_single(TUNE_ARCH, TUNE_WINDOW, seed=42, **kw))
    return pd.DataFrame(rows)

def plot_ofat(df, param, logx=False, fname=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df[param], df['RMSE (C)'], 'o-', color='darkorange', lw=2, label='RMSE (teste)')
    ax.axhline(BASELINE_RMSE, color='dimgrey', ls='--', lw=1.3,
               label=f'Baseline ({BASELINE_RMSE:.3f} C)')
    ax.fill_between(df[param], BASELINE_RMSE - NOISE_BAND, BASELINE_RMSE + NOISE_BAND,
                    color='grey', alpha=0.15, label=f'+/- {NOISE_BAND} C (ruido entre sementes)')
    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(param)
    ax.set_ylabel('Test RMSE (C)')
    ax.set_title(f'OFAT - {param}   ({TUNE_ARCH} @ {TUNE_WINDOW})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if fname:
        plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

print(f'Alvo do tuning : {TUNE_ARCH} @ {TUNE_WINDOW}')
print(f'Baseline       : {BASELINE}')

### 3.1 Referência (*baseline*)

Treina-se primeiro a configuração *baseline* (a do notebook principal) para obter a linha de referência `BASELINE_RMSE`, contra a qual todas as variações são comparadas nos gráficos seguintes.

In [ ]:
print('=== Baseline (GRU @ 24h, configuracao do notebook principal) ===')
baseline_row = run_single(TUNE_ARCH, TUNE_WINDOW, seed=42, **BASELINE)
BASELINE_RMSE = baseline_row['RMSE (C)']
print()
print(f'BASELINE_RMSE = {BASELINE_RMSE:.3f} C  (linha de referencia dos graficos)')

### 3.2 OFAT — número de unidades recorrentes (`units`)

Controla a **capacidade** da camada GRU. Mais unidades → mais parâmetros → maior capacidade de modelação, mas também maior custo e maior risco de *overfitting*. Procura-se o ponto onde aumentar a capacidade **deixa de compensar**.

In [ ]:
print('=== OFAT: units ===')
ofat_units = ofat('units', [32, 64, 128, 256])
plot_ofat(ofat_units, 'units', fname='tuning_ofat_units.png')
ofat_units[['units', 'params', 'epochs_run', 'RMSE (C)', 'MAE (C)', 'time_s']]

### 3.3 OFAT — taxa de aprendizagem (`learning_rate`)

É tipicamente o hiperparâmetro **mais sensível**. Demasiado baixa → convergência lenta (pode parar cedo por *early stopping* sem ter convergido); demasiado alta → treino instável/divergente. Eixo-x em escala logarítmica.

In [ ]:
print('=== OFAT: learning_rate ===')
ofat_lr = ofat('lr', [3e-4, 1e-3, 3e-3, 1e-2])
plot_ofat(ofat_lr, 'lr', logx=True, fname='tuning_ofat_lr.png')
ofat_lr[['lr', 'epochs_run', 'RMSE (C)', 'MAE (C)', 'time_s']]

### 3.4 OFAT — *dropout*

Regularização: desliga aleatoriamente uma fração das ativações em treino, reduzindo o *overfitting*. `0.0` = sem regularização; valores altos (`0.5`) podem causar *underfitting*. Procura-se o equilíbrio.

In [ ]:
print('=== OFAT: dropout ===')
ofat_drop = ofat('dropout', [0.0, 0.1, 0.2, 0.3, 0.5])
plot_ofat(ofat_drop, 'dropout', fname='tuning_ofat_dropout.png')
ofat_drop[['dropout', 'epochs_run', 'RMSE (C)', 'MAE (C)', 'time_s']]

### 3.5 OFAT — `batch_size`

Afeta a **estabilidade** do gradiente e a **velocidade** por época. *Batches* pequenos → atualizações mais ruidosas (por vezes melhor generalização, mais lento); grandes → mais estáveis e rápidos por época, mas podem precisar de mais épocas ou de `lr` maior.

In [ ]:
print('=== OFAT: batch_size ===')
ofat_bs = ofat('batch_size', [64, 128, 256, 512])
plot_ofat(ofat_bs, 'batch_size', fname='tuning_ofat_batch.png')
ofat_bs[['batch_size', 'epochs_run', 'RMSE (C)', 'MAE (C)', 'time_s']]

### 3.6 Grid search — `units` × `learning_rate`

O OFAT vê cada fator **isoladamente** e pode falhar **interações** (ex.: mais unidades só ajudam com uma `lr` mais baixa). Faz-se aqui um varrimento exaustivo do produto cartesiano dos dois fatores tipicamente mais influentes, visualizado como *heatmap* (verde = melhor RMSE).

> 3 × 3 = **9 treinos**. Ajusta as listas `grid_units` / `grid_lr` se quiseres mais/menos resolução.

In [ ]:
print('=== Grid search: units x learning_rate ===')
grid_units = [32, 64, 128]
grid_lr    = [3e-4, 1e-3, 3e-3]

grid_rows = []
for u in grid_units:
    for lr in grid_lr:
        kw = dict(BASELINE); kw['units'] = u; kw['lr'] = lr
        grid_rows.append(run_single(TUNE_ARCH, TUNE_WINDOW, seed=42, **kw))
grid_df = pd.DataFrame(grid_rows)

pivot = grid_df.pivot(index='units', columns='lr', values='RMSE (C)')
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax,
            cbar_kws={'label': 'Test RMSE (C)'}, linewidths=0.5)
ax.set_title(f'Grid search - RMSE (C)   ({TUNE_ARCH} @ {TUNE_WINDOW})')
ax.set_xlabel('learning_rate'); ax.set_ylabel('units')
plt.tight_layout(); plt.savefig('tuning_grid_units_lr.png', dpi=150, bbox_inches='tight'); plt.show()
pivot

### 3.7 Random search — espaço completo

A pesquisa em grelha cresce exponencialmente com o número de fatores. Para explorar os **5 hiperparâmetros em simultâneo** usa-se *random search*: amostram-se `N_TRIALS` combinações aleatórias do espaço. A literatura (Bergstra & Bengio, 2012) mostra que, para o mesmo orçamento, a pesquisa aleatória encontra tipicamente boas configurações **mais depressa** que a grelha, porque não desperdiça orçamento a varrer dimensões pouco relevantes.

A amostragem usa uma semente fixa (`random.Random(2024)`), portanto é **reprodutível**.

In [ ]:
print('=== Random search ===')
SEARCH_SPACE = {
    'units':       [32, 64, 96, 128, 192, 256],
    'dropout':     [0.0, 0.1, 0.2, 0.3, 0.4],
    'lr':          [3e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    'batch_size':  [64, 128, 256, 512],
    'dense_units': [16, 32, 64],
}
N_TRIALS = 15
rng = random.Random(2024)   # amostragem reprodutivel

seen, rand_rows = set(), []
while len(rand_rows) < N_TRIALS:
    cfg = {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}
    key = tuple(sorted(cfg.items()))
    if key in seen:            # evita repetir combinacoes
        continue
    seen.add(key)
    print(f'Trial {len(rand_rows)+1:>2}/{N_TRIALS}: {cfg}')
    rand_rows.append(run_single(TUNE_ARCH, TUNE_WINDOW, seed=42, **cfg))

rand_df = pd.DataFrame(rand_rows).sort_values('RMSE (C)').reset_index(drop=True)
print()
print('--- Random search ordenado por RMSE ---')
rand_df[['units', 'dropout', 'lr', 'batch_size', 'dense_units',
         'params', 'epochs_run', 'RMSE (C)', 'MAE (C)']]

---
## 4. Validação da melhor configuração (multi-semente)

Junta-se o resultado de **todas** as pesquisas, escolhe-se a configuração com menor RMSE (semente única) e revalida-se com **3 sementes**, comparando com o *baseline* também a 3 sementes.

**A pergunta decisiva:** o ganho do *tuning* é **maior** que a soma dos desvios-padrão entre sementes? Se não for, a configuração "ótima" pode ter ganho a *lottery* da inicialização e o ganho **não é real**.

In [ ]:
# Junta todas as pesquisas e escolhe a melhor configuracao (semente unica)
all_search = pd.concat([ofat_units, ofat_lr, ofat_drop, ofat_bs, grid_df, rand_df],
                       ignore_index=True)
best_row = all_search.sort_values('RMSE (C)').iloc[0]
best_cfg = {'units': int(best_row['units']), 'dropout': float(best_row['dropout']),
            'lr': float(best_row['lr']), 'batch_size': int(best_row['batch_size']),
            'dense_units': int(best_row['dense_units'])}
print(f'Melhor config (1 semente): {best_cfg}')
print(f'  -> RMSE = {best_row["RMSE (C)"]:.3f} C   vs baseline {BASELINE_RMSE:.3f} C')

In [ ]:
def run_multiseed(cfg, seeds=SEEDS):
    rows = [run_single(TUNE_ARCH, TUNE_WINDOW, seed=s, **cfg) for s in seeds]
    r = pd.DataFrame(rows)
    return r['RMSE (C)'].mean(), r['RMSE (C)'].std(), r

print('=== Melhor config tunada (3 sementes) ===')
best_mean, best_std, _ = run_multiseed(best_cfg)
print()
print('=== Baseline (3 sementes) ===')
base_mean, base_std, _ = run_multiseed(dict(BASELINE))

gain  = base_mean - best_mean
noise = base_std + best_std
print()
print('=' * 56)
print(f'Baseline   : {base_mean:.3f} +/- {base_std:.3f} C')
print(f'Best tuned : {best_mean:.3f} +/- {best_std:.3f} C')
print(f'Ganho      : {gain:+.3f} C   |   banda de ruido ~ {noise:.3f} C')
veredito = 'NAO E' if abs(gain) < noise else 'E'
print(f'=> O ganho do tuning {veredito} distinguivel do ruido entre sementes.')
print('=' * 56)

---
## 5. Conclusões & como usar no relatório

> **Nota:** os números concretos abaixo só ficam preenchidos **depois de correres o notebook**. Os espaços `[...]` são para o teu colega completar a partir das tabelas/figuras geradas.

### O que foi feito (resumo para o relatório)
Expandiu-se a otimização de hiperparâmetros do projeto, fixando a arquitetura líder (**GRU @ 24 h**) e pesquisando 5 hiperparâmetros (`units`, `learning_rate`, `dropout`, `batch_size`, `dense_units`) em **três níveis crescentes de sofisticação**:
1. **OFAT** — sensibilidade isolada de cada fator (Secção 3.2–3.5);
2. **Grid search** `units × learning_rate` — interações entre os dois fatores principais (Secção 3.6);
3. **Random search** — exploração conjunta do espaço completo, 15 ensaios (Secção 3.7).

A melhor configuração foi **revalidada com 3 sementes** e comparada honestamente com o *baseline* usando a **banda de ruído de inicialização** como critério de significância (Secção 4).

### Resultados a reportar (preencher após execução)
- Hiperparâmetro **mais sensível**: `[units / lr / dropout / batch_size]` (o que mais moveu o RMSE no OFAT).
- Melhor configuração encontrada: `units=[...]`, `lr=[...]`, `dropout=[...]`, `batch_size=[...]`, `dense_units=[...]`.
- RMSE *baseline* = `[...]` °C  →  RMSE tunado = `[...]` °C  (ganho = `[...]` °C).
- **Veredito de significância:** o ganho `[é / não é]` superior à banda de ruído (~`[...]` °C).

### Leitura esperada (alinhada com o notebook principal)
O notebook principal já concluiu que, neste problema (horizonte de 24 h, dominado pelos ciclos diário e sazonal já fornecidos como *features* cíclicas), as escolhas de arquitetura e janela produzem diferenças **dentro do ruído**. É expectável que o *tuning* fino confirme o mesmo padrão: ganhos **marginais**, frequentemente dentro da banda de ruído — o que é, em si, uma **conclusão metodológica válida e honesta** (o modelo está perto do limite de previsibilidade do problema, não mal afinado). Se algum hiperparâmetro produzir um ganho **acima** da banda de ruído, essa é a configuração a recomendar.

### Extensões possíveis (se houver tempo/compute)
- **KerasTuner** — implementado no **Apêndice (Secção 6)** como pesquisa automática/guiada (`BayesianOptimization`); fica desativado por omissão (requer `pip install keras-tuner`).
- Incluir o **tamanho da janela** (`24h`/`48h`) e o **número de camadas recorrentes** no espaço de pesquisa.
- Repetir a melhor configuração sobre as outras arquiteturas (LSTM/SimpleRNN) para confirmar que o ótimo de hiperparâmetros é estável entre células.


---
## 6. Apêndice (opcional) — KerasTuner (pesquisa automática)

Esta secção é **opcional** e **complementar** à pesquisa manual das Secções 3–4. Mostra como o mesmo problema se resolve com uma biblioteca dedicada de *hyperparameter tuning*, o **KerasTuner**, que **automatiza** a procura e usa uma estratégia **guiada** (`BayesianOptimization`): em vez de amostrar às cegas, modela a relação hiperparâmetros → desempenho e escolhe os próximos ensaios de forma informada.

**Diferenças face à pesquisa manual:**
- A pesquisa manual (OFAT/grid/random) é **transparente e didática** — vê-se exatamente o que cada fator faz. É a que deve liderar o relatório.
- O KerasTuner é **mais eficiente** para encontrar bons ótimos, mas funciona como **caixa-preta**. Serve para **confirmar** que a pesquisa manual não deixou ganhos óbvios por explorar.

**Notas:**
- Requer instalação: `pip install keras-tuner`. A célula está protegida por `try/except` — se a biblioteca não existir, apenas imprime um aviso e não falha.
- O objetivo otimizado é a **perda de validação** (`val_loss`); tunam-se `units`, `dropout`, `learning_rate` e `dense_units` (o `batch_size` fica fixo em 256 para simplificar). No fim, a melhor configuração é **reavaliada em teste, em °C**, e comparada com o *baseline*.
- Cria a pasta `kt_dir/` com os registos da pesquisa (pode ser apagada / adicionada ao `.gitignore`).
- **Custo:** `max_trials=12` ensaios. Reduz se precisares de um teste rápido.

In [ ]:
# Import protegido: nao quebra o notebook se o KerasTuner nao estiver instalado.
try:
    import keras_tuner as kt
    HAVE_KT = True
    print(f'KerasTuner {kt.__version__} disponivel.')
except ImportError:
    HAVE_KT = False
    print('KerasTuner NAO instalado.  Para usar esta seccao:  pip install keras-tuner')


def build_hypermodel(hp):
    # Mesma topologia do build_model, mas os hiperparametros sao definidos
    # pelo objeto `hp` do KerasTuner (que controla a pesquisa).
    units       = hp.Int('units', min_value=32, max_value=256, step=32)
    dropout     = hp.Float('dropout', min_value=0.0, max_value=0.4, step=0.1)
    lr          = hp.Choice('lr', [3e-4, 5e-4, 1e-3, 2e-3, 3e-3])
    dense_units = hp.Choice('dense_units', [16, 32, 64])

    window = datasets[TUNE_WINDOW]['window']
    model = Sequential(name='GRU_kt')
    model.add(layers.GRU(units, input_shape=(window, N_FEATURES), return_sequences=False))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='mse', metrics=['mae'])
    return model

In [ ]:
if HAVE_KT:
    ds = datasets[TUNE_WINDOW]
    set_seeds(42)

    tuner = kt.BayesianOptimization(
        build_hypermodel, objective='val_loss', max_trials=12,
        executions_per_trial=1, overwrite=True,
        directory='kt_dir', project_name='gru_24h')

    es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)
    print('=== KerasTuner: BayesianOptimization (objetivo = val_loss) ===')
    tuner.search(ds['X_train'], ds['y_train'],
                 validation_data=(ds['X_val'], ds['y_val']),
                 epochs=40, batch_size=256, callbacks=[es], verbose=0)

    best_hp = tuner.get_best_hyperparameters(1)[0]
    print()
    print('Melhores hiperparametros (KerasTuner):', best_hp.values)

    # Reconstroi e re-treina de forma limpa para avaliar em teste (em C).
    set_seeds(42)
    kt_model = build_hypermodel(best_hp)
    train_model(kt_model, ds, epochs=EPOCHS_TUNE, batch_size=256)
    kt_met = evaluate_model(kt_model, ds)

    print()
    print('=' * 56)
    print(f"KerasTuner best : RMSE = {kt_met['rmse_c']:.3f} C   MAE = {kt_met['mae_c']:.3f} C")
    print(f"Baseline        : RMSE = {BASELINE_RMSE:.3f} C")
    print(f"Diferenca       : {BASELINE_RMSE - kt_met['rmse_c']:+.3f} C   "
          f"(banda de ruido ~ {NOISE_BAND} C)")
    print('=' * 56)
else:
    print('Seccao ignorada (KerasTuner nao instalado).')